# Lab Practical - 4: CNN Architectures for Imbalanced Image Classification

This notebook implements the solution for Lab 4, covering:
1.  **Architecture Design**: ResNet18 vs MobileNetV2.
2.  **Imbalanced Handling**: Oversampling, Class Weighting.
3.  **Loss Functions**: CrossEntropy vs Focal Loss.
4.  **Datasets**: Imbalanced CIFAR-10 and Flower Photos.


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
import os
import random
from tqdm import tqdm
from torchvision.datasets.utils import download_and_extract_archive

# Set seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


In [13]:
class ImbalancedCIFAR10(torchvision.datasets.CIFAR10):
    def __init__(self, root, train=True, transform=None, target_transform=None, download=False, imbalance_ratio=0.01):
        super(ImbalancedCIFAR10, self).__init__(root, train, transform, target_transform, download)
        if train:
            self.imbalance_ratio = imbalance_ratio
            self.gen_imbalanced_data(10)

    def gen_imbalanced_data(self, cls_num):
        img_max = len(self.data) / cls_num
        img_num_per_cls = []
        for cls_idx in range(cls_num):
            num = img_max * (self.imbalance_ratio**(cls_idx / (cls_num - 1.0)))
            img_num_per_cls.append(int(num))
        
        new_data = []
        new_targets = []
        targets_np = np.array(self.targets)
        classes = np.unique(targets_np)
        self.cls_num_list = img_num_per_cls

        print(f'Imbalanced class distribution: {img_num_per_cls}')

        for the_class, the_img_num in zip(classes, img_num_list := img_num_per_cls):
            idx = np.where(targets_np == the_class)[0]
            np.random.shuffle(idx)
            selec_idx = idx[:the_img_num]
            new_data.append(self.data[selec_idx, ...])
            new_targets.extend([the_class, ] * the_img_num)
            
        self.data = np.vstack(new_data)
        self.targets = new_targets
        
    def get_cls_num_list(self):
        return self.cls_num_list

In [14]:
def get_flower_dataset(root='./data', transform=None):
    url = 'http://download.tensorflow.org/example_images/flower_photos.tgz'
    try:
        download_and_extract_archive(url, root, filename='flower_photos.tgz')
    except Exception as e:
        print(f'Download might have failed or file exists: {e}')
    dataset_path = os.path.join(root, 'flower_photos')
    # ImageFolder expects subdirectories for classes, which flower_photos has
    dataset = torchvision.datasets.ImageFolder(root=dataset_path, transform=transform)
    return dataset

In [15]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = nn.CrossEntropyLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [16]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=5, name='Model'):
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    print(f'Starting training for {name}...')
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        train_loss = running_loss / len(train_loader)
        train_acc = 100. * correct / total
        
        # Validation
        val_loss, val_acc, _, _ = evaluate_model(model, val_loader, criterion)
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        print(f'Epoch {epoch+1}: Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        
    return history

def evaluate_model(model, loader, criterion=None):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_preds = []
    
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            if criterion:
                loss = criterion(outputs, labels)
                running_loss += loss.item()
            
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            
    loss = running_loss / len(loader) if criterion else 0
    acc = 100. * correct / total
    return loss, acc, np.array(all_labels), np.array(all_preds)

def plot_history(history, title='Training History'):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title('Accuracy')
    plt.legend()
    plt.suptitle(title)
    plt.show()

def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

In [ ]:
# Transforms
transform_train = transforms.Compose([
    transforms.Resize((32, 32)), # CIFAR size
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Prepare Data
trainset_imb = ImbalancedCIFAR10(root='./data', train=True, download=True, transform=transform_train, imbalance_ratio=0.01)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
trainloader = DataLoader(trainset_imb, batch_size=64, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

 26%|██▌       | 44.5M/170M [26:43<40:22, 52.0kB/s]  

In [ ]:
print('--- Experiment 1: Baseline ResNet18 (CrossEntropy) ---')
model_base = models.resnet18(pretrained=True)
model_base.fc = nn.Linear(model_base.fc.in_features, 10)
model_base = model_base.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_base.parameters(), lr=0.001)

hist_base = train_model(model_base, trainloader, testloader, criterion, optimizer, num_epochs=5, name='Baseline')
plot_history(hist_base, 'Baseline ResNet18')

In [ ]:
print('--- Experiment 2: Oversampling (WeightedRandomSampler) ---')
# Calculate weights
cls_num_list = trainset_imb.get_cls_num_list()
cls_weights = [1.0 / x for x in cls_num_list]
sample_weights = [cls_weights[t] for t in trainset_imb.targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

trainloader_sampler = DataLoader(trainset_imb, batch_size=64, sampler=sampler, num_workers=2)

model_os = models.resnet18(pretrained=True)
model_os.fc = nn.Linear(model_os.fc.in_features, 10)
model_os = model_os.to(device)

optimizer = optim.Adam(model_os.parameters(), lr=0.001)
hist_os = train_model(model_os, trainloader_sampler, testloader, criterion, optimizer, num_epochs=5, name='Oversampling')
plot_history(hist_os, 'Oversampling ResNet18')

In [ ]:
print('--- Experiment 3: Class Weighted Loss ---')
weights = torch.tensor(cls_weights).float().to(device)
weights = weights / weights.sum() * 10 # Normalize roughly
criterion_weighted = nn.CrossEntropyLoss(weight=weights)

model_cw = models.resnet18(pretrained=True)
model_cw.fc = nn.Linear(model_cw.fc.in_features, 10)
model_cw = model_cw.to(device)

optimizer = optim.Adam(model_cw.parameters(), lr=0.001)
hist_cw = train_model(model_cw, trainloader, testloader, criterion_weighted, optimizer, num_epochs=5, name='Class Weighted')
plot_history(hist_cw, 'Class Weighted ResNet18')

In [ ]:
print('--- Experiment 4: Focal Loss ---')
criterion_focal = FocalLoss(gamma=2.0, alpha=1.0)

model_fl = models.resnet18(pretrained=True)
model_fl.fc = nn.Linear(model_fl.fc.in_features, 10)
model_fl = model_fl.to(device)

optimizer = optim.Adam(model_fl.parameters(), lr=0.001)
hist_fl = train_model(model_fl, trainloader, testloader, criterion_focal, optimizer, num_epochs=5, name='Focal Loss')
plot_history(hist_fl, 'Focal Loss ResNet18')

In [ ]:
print('--- Experiment 5: MobileNetV2 Comparison ---')
model_mob = models.mobilenet_v2(pretrained=True)
model_mob.classifier[1] = nn.Linear(model_mob.last_channel, 10)
model_mob = model_mob.to(device)

# Use the best strategy (e.g., Oversampling) or just baseline for strict architecture compare
# Let's use Oversampling as it likely performed well
optimizer = optim.Adam(model_mob.parameters(), lr=0.001)
hist_mob = train_model(model_mob, trainloader_sampler, testloader, criterion, optimizer, num_epochs=5, name='MobileNetV2 (Oversampled)')
plot_history(hist_mob, 'MobileNetV2')

In [ ]:
print('--- Experiment 6: Flowers Dataset (Dataset 2) ---')
# Note: This might take a while to download
transform_flower = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

try:
    flower_set = get_flower_dataset(transform=transform_flower)
    # Split
    train_size = int(0.8 * len(flower_set))
    test_size = len(flower_set) - train_size
    train_flower, test_flower = torch.utils.data.random_split(flower_set, [train_size, test_size])
    
    trainloader_flower = DataLoader(train_flower, batch_size=32, shuffle=True, num_workers=2)
    testloader_flower = DataLoader(test_flower, batch_size=32, shuffle=False, num_workers=2)
    
    model_flower = models.resnet18(pretrained=True)
    model_flower.fc = nn.Linear(model_flower.fc.in_features, 5) # 5 classes
    model_flower = model_flower.to(device)
    
    optimizer = optim.Adam(model_flower.parameters(), lr=0.001)
    train_model(model_flower, trainloader_flower, testloader_flower, criterion, optimizer, num_epochs=3, name='Flowers ResNet')
    
except Exception as e:
    print(f'Could not run Flowers experiment: {e}')

In [ ]:
print('--- Final Evaluation & Confusion Matrices ---')
# Evaluate best model (e.g. Oversampling) vs Baseline
_, _, y_true_base, y_pred_base = evaluate_model(model_base, testloader)
_, _, y_true_os, y_pred_os = evaluate_model(model_os, testloader)

print('Baseline Report:')
print(classification_report(y_true_base, y_pred_base, target_names=classes))

print('Oversampling Report:')
print(classification_report(y_true_os, y_pred_os, target_names=classes))

plot_confusion_matrix(y_true_base, y_pred_base, classes, title='Baseline Confusion Matrix')
plot_confusion_matrix(y_true_os, y_pred_os, classes, title='Oversampling Confusion Matrix')